# Inversion of Stokes Profiles (SIR) — Implementation / 구현

**Paper**: Ruiz Cobo, B. & del Toro Iniesta, J. C., *The Astrophysical Journal*, **398**, 375 (1992). DOI: 10.1086/171862

**EN.** This notebook walks through a simplified self-contained SIR-like pipeline:

1. A **Milne-Eddington (ME) Stokes synthesis** — the fast analytical forward model
2. **Response functions** $R_m(\tau, \lambda) = \partial I(\lambda)/\partial x_m$ via finite differences for $B, \gamma, \phi$
3. A **Levenberg-Marquardt $\chi^2$-fit** recovering $(B, \gamma, \phi, v, \eta_0, \Delta\lambda_D)$ from noisy synthetic Stokes
4. A **node-based $T(\tau)$ profile** with 5 nodes in $\log\tau$, showing how node parameterization + cubic splines represent a depth-dependent atmosphere
5. A **covariance-matrix uncertainty** estimator from the converged Marquardt curvature matrix

**KR.** 본 노트북은 단순화한 자기완결적 SIR 유사 파이프라인을 단계별로 구현한다:

1. **Milne-Eddington (ME) 스토크스 합성** — 빠른 해석적 정방향 모델
2. **응답함수** $R_m(\tau, \lambda) = \partial I(\lambda)/\partial x_m$ — $B, \gamma, \phi$ 에 대해 차분법으로 계산
3. **Levenberg-Marquardt $\chi^2$ 최소화** — 노이즈가 있는 합성 스토크스에서 $(B, \gamma, \phi, v, \eta_0, \Delta\lambda_D)$ 복원
4. **노드 기반 $T(\tau)$ 프로파일** — $\log\tau$ 상 5개 노드와 큐빅 스플라인으로 깊이-의존 대기 표현
5. 수렴된 Marquardt 곡률행렬에서 **공분산 행렬을 통한 불확실도 추정**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

rng = np.random.default_rng(42)

## Part 1: Milne-Eddington Stokes Synthesis / ME 스토크스 합성

**EN.** In the Milne-Eddington approximation the source function is linear in optical depth, $S = S_0 + \mu \tau$, and the absorption matrix is assumed to be $\tau$-independent. This yields a closed-form analytical solution for the emergent Stokes vector (Unno 1956, Rachkovsky 1962, Landi Degl'Innocenti 1992). It is the standard baseline forward model for inversion codes (AHH, MILOS, SIR's initialization module).

**KR.** Milne-Eddington 근사에서는 source function이 광학심도의 선형함수 $S = S_0 + \mu \tau$이고 흡수행렬은 $\tau$-독립으로 가정한다. 이 경우 방출 스토크스 벡터의 해석적 폐쇄형이 존재한다 (Unno 1956; Rachkovsky 1962; Landi Degl'Innocenti 1992). AHH, MILOS, SIR의 초기화 모듈 모두가 채택하는 표준 기준 정방향 모델이다.

In [ ]:
def voigt_approx(a, u):
    """Approximate Voigt H and Faraday-Voigt F functions.

    Uses a simple Lorentzian+Gaussian combination — sufficient accuracy
    for the didactic ME inversion demo. Production SIR uses full Voigt-Faraday
    routines (Wittmann 1974).

    Args:
        a: Damping parameter (dimensionless).
        u: Frequency offset in Doppler widths.

    Returns:
        Tuple (H, F) of real and dispersion profiles.
    """
    H = np.exp(-u**2) + a / np.sqrt(np.pi) / (u**2 + a**2 + 1e-12)
    F = 0.5 * u / np.sqrt(np.pi) / (u**2 + a**2 + 1e-12)
    return H, F

In [ ]:
def me_stokes(wl, lam0, B, gamma_deg, phi_deg, v_los, eta0=5.0,
              dlam_D=30.0, a_damp=0.1, g_eff=2.5, S0=1.0, S1=3.0):
    """Synthesize Stokes I, Q, U, V using the Milne-Eddington solution.

    Zeeman splitting in the weak-field normal-triplet approximation.

    Args:
        wl: Wavelength array in mA (centered near line core, can be negative).
        lam0: Rest wavelength of the line in mA (for Zeeman splitting scale).
        B: Magnetic field strength in Gauss.
        gamma_deg: Magnetic field inclination in degrees (0 = along LOS).
        phi_deg: Magnetic field azimuth in degrees.
        v_los: Line-of-sight velocity in km/s (positive = redshift).
        eta0: Line-to-continuum absorption ratio.
        dlam_D: Doppler width in mA.
        a_damp: Damping parameter.
        g_eff: Effective Landé factor.
        S0: Source function at tau=0.
        S1: Gradient of source function.

    Returns:
        Tuple (I, Q, U, V), each array of shape (len(wl),) normalized by
        continuum S0 + S1.
    """
    c_light = 2.9979e5  # km/s
    # Zeeman splitting in mA (Lambda_B = 4.67e-13 * lam0^2 * B * g_eff)
    lamB = 4.67e-13 * (lam0 * 1e-3) ** 2 * B * g_eff * 1e11  # mA
    v_shift = v_los / c_light * lam0  # mA
    gamma = np.deg2rad(gamma_deg)
    phi = np.deg2rad(phi_deg)

    # Reduced frequencies for the three Zeeman components
    u_p = (wl - v_shift - lamB) / dlam_D  # sigma+
    u_m = (wl - v_shift + lamB) / dlam_D  # sigma-
    u_0 = (wl - v_shift) / dlam_D         # pi

    Hp, Fp = voigt_approx(a_damp, u_p)
    Hm, Fm = voigt_approx(a_damp, u_m)
    H0, F0 = voigt_approx(a_damp, u_0)

    # Absorption coefficients (normalized)
    eta_I = 1 + 0.5 * eta0 * (H0 * np.sin(gamma)**2 +
                               0.5 * (Hp + Hm) * (1 + np.cos(gamma)**2))
    eta_Q = 0.5 * eta0 * (H0 - 0.5 * (Hp + Hm)) * np.sin(gamma)**2 * np.cos(2*phi)
    eta_U = 0.5 * eta0 * (H0 - 0.5 * (Hp + Hm)) * np.sin(gamma)**2 * np.sin(2*phi)
    eta_V = 0.5 * eta0 * (Hp - Hm) * np.cos(gamma)

    rho_Q = 0.5 * eta0 * (F0 - 0.5 * (Fp + Fm)) * np.sin(gamma)**2 * np.cos(2*phi)
    rho_U = 0.5 * eta0 * (F0 - 0.5 * (Fp + Fm)) * np.sin(gamma)**2 * np.sin(2*phi)
    rho_V = 0.5 * eta0 * (Fp - Fm) * np.cos(gamma)

    # Unno-Rachkovsky denominator
    D = (eta_I**2 * (eta_I**2 - eta_Q**2 - eta_U**2 - eta_V**2
                     + rho_Q**2 + rho_U**2 + rho_V**2)
         - (eta_Q * rho_Q + eta_U * rho_U + eta_V * rho_V)**2) + 1e-30

    I_prof = S0 + S1 * eta_I * (eta_I**2 + rho_Q**2 + rho_U**2 + rho_V**2) / D
    Q_prof = -S1 * (eta_I**2 * eta_Q + eta_I * (eta_V * rho_U - eta_U * rho_V)
                    + rho_Q * (eta_Q * rho_Q + eta_U * rho_U + eta_V * rho_V)) / D
    U_prof = -S1 * (eta_I**2 * eta_U + eta_I * (eta_Q * rho_V - eta_V * rho_Q)
                    + rho_U * (eta_Q * rho_Q + eta_U * rho_U + eta_V * rho_V)) / D
    V_prof = -S1 * (eta_I**2 * eta_V + rho_V * (eta_Q * rho_Q + eta_U * rho_U
                                                + eta_V * rho_V)) / D

    # Normalize to continuum intensity
    I_c = S0 + S1
    return I_prof / I_c, Q_prof / I_c, U_prof / I_c, V_prof / I_c

In [ ]:
# Synthesize reference Fe I 6302.5 Stokes profiles
lam0 = 6302500.0  # mA
wl = np.linspace(-400, 400, 161)  # mA offsets from line center, 5 mA sampling

# True ("reference") parameters
true_params = {
    'B': 2000.0,       # Gauss
    'gamma': 45.0,     # degrees
    'phi': 30.0,       # degrees
    'v_los': 1.5,      # km/s
    'eta0': 5.0,
    'dlam_D': 30.0,    # mA
}

I_ref, Q_ref, U_ref, V_ref = me_stokes(wl, lam0, **true_params)

# Add noise: S/N = 250 in continuum
noise_level = 1.0 / 250.0
I_obs = I_ref + rng.normal(0, noise_level, size=wl.size)
Q_obs = Q_ref + rng.normal(0, noise_level, size=wl.size)
U_obs = U_ref + rng.normal(0, noise_level, size=wl.size)
V_obs = V_ref + rng.normal(0, noise_level, size=wl.size)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, ref, obs, title in zip(axes.flat,
                                [I_ref, Q_ref, U_ref, V_ref],
                                [I_obs, Q_obs, U_obs, V_obs],
                                ['Stokes $I/I_c$', 'Stokes $Q/I_c$',
                                 'Stokes $U/I_c$', 'Stokes $V/I_c$']):
    ax.plot(wl, ref, 'k-', lw=1.5, label='reference / 참조')
    ax.plot(wl, obs, 'ro', ms=2, label='noisy (S/N=250)')
    ax.set_xlabel('$\\Delta\\lambda$ (mÅ)')
    ax.set_ylabel(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
fig.suptitle('Fe I 6302.5 — ME synthesis (reference) + S/N=250 noise', fontsize=12)
fig.tight_layout()

## Part 2: Response Functions via Finite Differences / 차분법 응답함수

**EN.** In SIR's production code, RFs are evaluated analytically from the Sánchez Almeida (1992) expression alongside the DELO forward solution — a major speed-up. Here, for pedagogical simplicity, we compute RFs for the ME synthesis by finite differences:

$$R_m(\lambda) \approx \frac{I(\lambda; x_m + \Delta x) - I(\lambda; x_m - \Delta x)}{2\Delta x}$$

This illustrates the RF shape and its dependence on Stokes component and perturbed parameter.

**KR.** 실제 SIR 코드에서는 Sánchez Almeida (1992) 표현으로 DELO 정방향 풀이와 함께 해석적으로 RF를 계산하여 큰 속도 향상을 얻는다. 여기서는 교육 목적으로 ME 합성에 대해 차분법으로 RF를 계산한다:

$$R_m(\lambda) \approx \frac{I(\lambda; x_m + \Delta x) - I(\lambda; x_m - \Delta x)}{2\Delta x}$$

이로써 RF의 모양과 스토크스 성분 및 섭동 파라미터에 대한 의존성을 볼 수 있다.

In [ ]:
def response_function_fd(param_name, params, lam0, wl, dx_rel=0.01):
    """Compute finite-difference response functions of all four Stokes.

    Args:
        param_name: Name of the parameter to perturb ('B', 'gamma', 'phi', etc.).
        params: Dictionary of current parameter values.
        lam0: Rest wavelength (mA).
        wl: Wavelength offsets array (mA).
        dx_rel: Relative perturbation step.

    Returns:
        Tuple (R_I, R_Q, R_U, R_V), each shape (len(wl),).
    """
    x = params[param_name]
    dx = max(abs(x) * dx_rel, dx_rel)

    p_plus = dict(params); p_plus[param_name] = x + dx
    p_minus = dict(params); p_minus[param_name] = x - dx

    I_p, Q_p, U_p, V_p = me_stokes(wl, lam0, **p_plus)
    I_m, Q_m, U_m, V_m = me_stokes(wl, lam0, **p_minus)

    return ((I_p - I_m) / (2*dx),
            (Q_p - Q_m) / (2*dx),
            (U_p - U_m) / (2*dx),
            (V_p - V_m) / (2*dx))

In [ ]:
# Compute RFs for B, gamma, phi
RF_B = response_function_fd('B', true_params, lam0, wl)
RF_gamma = response_function_fd('gamma', true_params, lam0, wl)
RF_phi = response_function_fd('phi', true_params, lam0, wl)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
stokes_names = ['I', 'Q', 'U', 'V']
styles = ['-', ':', '--', '-.']

for ax, RF, title in zip(axes,
                          [RF_B, RF_gamma, RF_phi],
                          ['$R_B(\\lambda)$ — field strength',
                           '$R_\\gamma(\\lambda)$ — inclination',
                           '$R_\\phi(\\lambda)$ — azimuth']):
    norm = np.max(np.abs(RF[0])) + 1e-30
    for k, (R, name, ls) in enumerate(zip(RF, stokes_names, styles)):
        ax.plot(wl, R / norm, ls, lw=1.5, label=f'Stokes {name}')
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xlabel('$\\Delta\\lambda$ (mÅ)')
    ax.set_ylabel('Normalized RF / 정규화 RF')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
fig.suptitle('Response functions — ME forward model (finite differences)', fontsize=12)
fig.tight_layout()

**EN.** Note that the RF for Stokes $V$ to $B$ is bipolar around line center (the classic Zeeman signature), while RF for Stokes $I$ is largest near line core. Azimuth $\phi$ affects only $Q, U$ significantly — linear polarization is its primary carrier.

**KR.** Stokes $V$ 의 $B$ 에 대한 RF는 선 중심 부근에서 쌍극성(전형적 제만 signature), Stokes $I$ 의 RF는 선 중심에서 최대. 방위각 $\phi$ 는 $Q, U$ 에 주로 영향 — 선형편광이 주 정보 운반자.

## Part 3: Levenberg-Marquardt $\chi^2$-Fit / LM $\chi^2$ 최소화

**EN.** We now solve the inversion problem: given noisy observed Stokes $(I_\mathrm{obs}, Q_\mathrm{obs}, U_\mathrm{obs}, V_\mathrm{obs})$, recover $(B, \gamma, \phi, v_\mathrm{los}, \eta_0, \Delta\lambda_D)$.

Levenberg-Marquardt update: $(A + \lambda \mathrm{diag}(A))\,\delta\mathbf{a} = -\nabla\chi^2$, where $A_{jk} = \sum_i R_j(\lambda_i) R_k(\lambda_i)/\sigma_i^2$ is the curvature matrix.

**KR.** 이제 인버전 문제를 푼다: 관측 스토크스 $(I_\mathrm{obs}, Q_\mathrm{obs}, U_\mathrm{obs}, V_\mathrm{obs})$ 로부터 $(B, \gamma, \phi, v_\mathrm{los}, \eta_0, \Delta\lambda_D)$ 를 복원.

LM 업데이트: $(A + \lambda \mathrm{diag}(A))\,\delta\mathbf{a} = -\nabla\chi^2$, 여기서 $A_{jk} = \sum_i R_j(\lambda_i) R_k(\lambda_i)/\sigma_i^2$.

In [ ]:
def synth_all(params, lam0, wl):
    """Concatenate all four Stokes into a single vector.

    Args:
        params: Parameter dictionary.
        lam0: Rest wavelength (mA).
        wl: Wavelength offsets (mA).

    Returns:
        Flattened array of shape (4*len(wl),).
    """
    I, Q, U, V = me_stokes(wl, lam0, **params)
    return np.concatenate([I, Q, U, V])

In [ ]:
def build_jacobian(params, lam0, wl, free_names, dx_rel=0.01):
    """Build the Jacobian matrix d(stokes_vector) / d(params).

    Args:
        params: Current parameter dictionary.
        lam0: Rest wavelength (mA).
        wl: Wavelength offsets (mA).
        free_names: List of free parameter names.
        dx_rel: Relative perturbation step.

    Returns:
        Jacobian of shape (4*len(wl), len(free_names)).
    """
    N = 4 * wl.size
    J = np.zeros((N, len(free_names)))
    for k, name in enumerate(free_names):
        x = params[name]
        dx = max(abs(x) * dx_rel, 1e-3)
        p_plus = dict(params); p_plus[name] = x + dx
        p_minus = dict(params); p_minus[name] = x - dx
        f_plus = synth_all(p_plus, lam0, wl)
        f_minus = synth_all(p_minus, lam0, wl)
        J[:, k] = (f_plus - f_minus) / (2*dx)
    return J

In [ ]:
def levenberg_marquardt(params0, lam0, wl, obs_vec, sigma, free_names,
                         max_iter=50, tol=1e-6):
    """Levenberg-Marquardt nonlinear least-squares fit.

    Args:
        params0: Initial parameter dictionary.
        lam0: Rest wavelength (mA).
        wl: Wavelength offsets (mA).
        obs_vec: Flattened observed Stokes (4*len(wl),).
        sigma: Noise standard deviation (scalar, assumes uniform).
        free_names: List of parameter names to fit.
        max_iter: Maximum iteration count.
        tol: Convergence tolerance on chi-squared fractional change.

    Returns:
        Tuple (params_best, chi2_history, covariance_matrix).
    """
    params = dict(params0)
    lam_damp = 1e-3
    chi2_hist = []

    def chi2(p):
        r = (obs_vec - synth_all(p, lam0, wl)) / sigma
        return np.sum(r**2)

    chi2_cur = chi2(params)
    chi2_hist.append(chi2_cur)

    for it in range(max_iter):
        J = build_jacobian(params, lam0, wl, free_names) / sigma
        r = (obs_vec - synth_all(params, lam0, wl)) / sigma
        A = J.T @ J
        b = J.T @ r
        A_damped = A + lam_damp * np.diag(np.diag(A))
        try:
            delta = np.linalg.solve(A_damped, b)
        except np.linalg.LinAlgError:
            lam_damp *= 10
            continue

        trial = dict(params)
        for k, name in enumerate(free_names):
            trial[name] = params[name] + delta[k]
        chi2_trial = chi2(trial)

        if chi2_trial < chi2_cur:
            frac = (chi2_cur - chi2_trial) / max(chi2_cur, 1e-30)
            params = trial
            chi2_cur = chi2_trial
            chi2_hist.append(chi2_cur)
            lam_damp = max(lam_damp / 10, 1e-8)
            if frac < tol:
                break
        else:
            lam_damp = min(lam_damp * 10, 1e8)

    # Covariance from converged curvature matrix
    J_final = build_jacobian(params, lam0, wl, free_names) / sigma
    A_final = J_final.T @ J_final
    try:
        cov = np.linalg.inv(A_final)
    except np.linalg.LinAlgError:
        cov = np.full_like(A_final, np.nan)
    return params, np.array(chi2_hist), cov

In [ ]:
# Initial guess — deliberately offset from truth
params0 = {
    'B': 1000.0,
    'gamma': 20.0,
    'phi': 0.0,
    'v_los': 0.0,
    'eta0': 3.0,
    'dlam_D': 25.0,
}
free_names = ['B', 'gamma', 'phi', 'v_los', 'eta0', 'dlam_D']

obs_vec = np.concatenate([I_obs, Q_obs, U_obs, V_obs])

params_fit, chi2_hist, cov = levenberg_marquardt(
    params0, lam0, wl, obs_vec, noise_level, free_names, max_iter=60
)

print('Marquardt converged in', len(chi2_hist)-1, 'iterations.')
print()
print(f"{'Parameter':<10} {'True':>10} {'Initial':>10} {'Recovered':>12} {'1-sigma':>10}")
for k, name in enumerate(free_names):
    sig = np.sqrt(cov[k, k])
    print(f'{name:<10} {true_params[name]:>10.3f} {params0[name]:>10.3f} '
          f'{params_fit[name]:>12.3f} {sig:>10.3f}')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(chi2_hist, 'o-')
ax.set_xlabel('Iteration / 반복')
ax.set_ylabel('$\\chi^2$')
ax.set_title('Levenberg-Marquardt $\\chi^2$ convergence')
ax.grid(alpha=0.3)

In [ ]:
# Compare fit with observations
I_fit, Q_fit, U_fit, V_fit = me_stokes(wl, lam0, **params_fit)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, ref, obs, fit, title in zip(axes.flat,
                                     [I_ref, Q_ref, U_ref, V_ref],
                                     [I_obs, Q_obs, U_obs, V_obs],
                                     [I_fit, Q_fit, U_fit, V_fit],
                                     ['Stokes $I/I_c$', 'Stokes $Q/I_c$',
                                      'Stokes $U/I_c$', 'Stokes $V/I_c$']):
    ax.plot(wl, ref, 'k--', lw=1, alpha=0.7, label='reference')
    ax.plot(wl, obs, 'ro', ms=2, alpha=0.5, label='noisy obs')
    ax.plot(wl, fit, 'b-', lw=1.5, label='LM fit')
    ax.set_xlabel('$\\Delta\\lambda$ (mÅ)')
    ax.set_ylabel(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
fig.suptitle('Recovered Stokes profiles (LM fit) vs reference and noisy input', fontsize=12)
fig.tight_layout()

## Part 4: Node-based $T(\tau)$ Profile / 노드 기반 $T(\tau)$ 프로파일

**EN.** SIR represents each depth-dependent physical magnitude by its values at $m$ equi-spaced nodes in $\log\tau$, interpolated with cubic splines (Appendix A of the paper). Here we illustrate the concept for temperature $T(\tau)$ with 5 nodes.

**KR.** SIR은 각 깊이-의존 물리량을 $\log\tau$ 상 $m$ 개 등간격 노드 값으로 기술하고 큐빅 스플라인으로 보간한다 (논문 부록 A). 여기서는 온도 $T(\tau)$ 에 대해 5 노드로 개념을 시연한다.

In [ ]:
def t_from_nodes(log_tau_grid, node_log_tau, node_T):
    """Reconstruct T(log_tau) from node values via cubic spline.

    Args:
        log_tau_grid: Dense log(tau) grid (e.g., 40 points).
        node_log_tau: Equi-spaced node positions in log(tau), length m.
        node_T: Temperature values at the nodes, length m.

    Returns:
        Temperature profile on log_tau_grid.
    """
    cs = CubicSpline(node_log_tau, node_T)
    return cs(log_tau_grid)

In [ ]:
# Dense log(tau) grid of 40 points (as in SIR Fig. 1)
log_tau_grid = np.linspace(-4.0, 1.0, 40)

# Reference umbral-like temperature (roughly Maltby et al. 1986 model E)
def ref_T(logt):
    return 3800 + 400 * np.exp(2*logt) + 2500 * (1/(1+np.exp(-2*(logt-0)) ))
T_ref = ref_T(log_tau_grid)

# Sample at 5 equi-spaced nodes and interpolate
m = 5
node_log_tau = np.linspace(log_tau_grid[0], log_tau_grid[-1], m)
node_T_true = ref_T(node_log_tau)
T_node_interp = t_from_nodes(log_tau_grid, node_log_tau, node_T_true)

# Same with m=2 (linear-like) for contrast
m2 = 2
node_log_tau_2 = np.linspace(log_tau_grid[0], log_tau_grid[-1], m2)
node_T_2 = ref_T(node_log_tau_2)
T_node_interp_2 = np.interp(log_tau_grid, node_log_tau_2, node_T_2)

# Same with m=9
m9 = 9
node_log_tau_9 = np.linspace(log_tau_grid[0], log_tau_grid[-1], m9)
node_T_9 = ref_T(node_log_tau_9)
T_node_interp_9 = t_from_nodes(log_tau_grid, node_log_tau_9, node_T_9)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(log_tau_grid, T_ref, 'k-', lw=2, label='reference $T(\\tau)$')
ax.plot(log_tau_grid, T_node_interp_2, 'r--', lw=1.5, label=f'$m=2$ nodes (linear)')
ax.plot(log_tau_grid, T_node_interp, 'b-.', lw=1.5, label=f'$m={m}$ nodes (cubic spline)')
ax.plot(log_tau_grid, T_node_interp_9, 'g:', lw=2, label=f'$m={m9}$ nodes (cubic spline)')
ax.plot(node_log_tau, node_T_true, 'bs', ms=9, label='$m=5$ node values')
ax.set_xlabel('$\\log \\tau$')
ax.set_ylabel('$T$ (K)')
ax.set_title('Node-based parameterization of $T(\\tau)$ — SIR Appendix A')
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()

print(f'RMS ($m=2$): {np.sqrt(np.mean((T_ref - T_node_interp_2)**2)):.1f} K')
print(f'RMS ($m=5$): {np.sqrt(np.mean((T_ref - T_node_interp)**2)):.1f} K')
print(f'RMS ($m=9$): {np.sqrt(np.mean((T_ref - T_node_interp_9)**2)):.1f} K')

**EN.** Moving from $m=2$ to $m=5$ to $m=9$ nodes dramatically improves the representation of $T(\tau)$. SIR's progressive refinement schedule exactly follows this logic: start coarse, converge, refine.

**KR.** $m=2 \to 5 \to 9$ 로 노드를 늘리면 $T(\tau)$ 표현이 급격히 개선된다. SIR의 점진적 정제 전략이 바로 이 논리를 따른다: 거친 모델에서 시작 → 수렴 → 정제.

## Part 5: Uncertainty Estimation from the Covariance Matrix / 공분산 행렬을 통한 불확실도 추정

**EN.** At convergence the Marquardt damping vanishes ($\lambda \to 0$), so $A^{-1}$ becomes the parameter covariance matrix. Diagonal elements give the parameter variances; off-diagonals reveal degeneracies. Below we visualize the correlation matrix of our recovered parameters.

**KR.** 수렴 시 Marquardt 감쇠가 소실($\lambda \to 0$)되므로 $A^{-1}$ 이 파라미터 공분산 행렬이 된다. 대각 원소가 파라미터 분산, 비대각 원소가 퇴화(degeneracy)를 보여준다. 복원된 파라미터의 상관행렬을 시각화한다.

In [ ]:
# Correlation matrix
sigmas = np.sqrt(np.diag(cov))
corr = cov / np.outer(sigmas, sigmas)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(free_names)))
ax.set_xticklabels(free_names, rotation=30)
ax.set_yticks(range(len(free_names)))
ax.set_yticklabels(free_names)
for i in range(len(free_names)):
    for j in range(len(free_names)):
        ax.text(j, i, f'{corr[i, j]:+.2f}', ha='center', va='center',
                color='black' if abs(corr[i, j]) < 0.6 else 'white', fontsize=9)
plt.colorbar(im, ax=ax, label='correlation')
ax.set_title('Parameter correlation matrix at convergence')
fig.tight_layout()

print('Recovered 1-sigma uncertainties:')
for k, name in enumerate(free_names):
    print(f'  {name}: {params_fit[name]:.3f} ± {sigmas[k]:.3f} '
          f'(truth: {true_params[name]:.3f})')

## Summary / 요약

| Concept / 개념 | This Paper (SIR 1992) / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Forward model / 정방향 모델 | DELO method (Rees+89) on full RTE / 완전 RTE의 DELO 법 | NICOLE, STiC (non-LTE, PRD) |
| Response function computation / RF 계산 | Analytical (Sánchez Almeida 1992) / 해석적 | Same formalism, auto-differentiation in ML emulators |
| Node parameterization / 노드 매개변수화 | $m=2, 5, 9$ equi-spaced + cubic spline | Same; sometimes radial-basis or adaptive nodes |
| Minimizer / 최적화기 | Marquardt-Levenberg | Trust-region, Adam (for ML surrogates) |
| Matrix regularization / 행렬 정규화 | Modified SVD / 수정 SVD | Tikhonov, L1/L2 priors |
| Uncertainty / 불확실도 | $\mathbf{A}^{-1}$ covariance / 공분산 | MCMC (Asensio Ramos), Bayesian ensembles |
| Typical CPU time / 전형 CPU 시간 | 50 min (4×250, 4 Stokes) on Eclipse 20000 | <10 s per pixel on modern CPU; ms on GPU emulator |

### Key Numerical Takeaways / 핵심 수치 요약

**EN.**
- 6-line full-Stokes inversion recovers $T$ to rms 10 K, $B$ to 70 G, $v$ to 0.04 km/s at S/N=250.
- 2-line full-Stokes inversion (Fe I 6301.5/6302.5): rms $T$=30 K, $B$=100 G, $\gamma$=3°, $\phi$=7°.
- 2-line $I, V$-only inversion still recovers $\phi$ to ~15° — magneto-optical coupling carries azimuth into $V$.
- Information-rich range $\log\tau \in [-2.6, 0.2]$ for six lines; shrinks to $\log\tau < -0.4$ for two lines.

**KR.**
- 6선 풀-스토크스 인버전: S/N=250에서 $T$ rms 10 K, $B$ 70 G, $v$ 0.04 km/s.
- 2선 풀-스토크스 (Fe I 6301.5/6302.5): rms $T$=30 K, $B$=100 G, $\gamma$=3°, $\phi$=7°.
- 2선 $I, V$ 만: $\phi$ 를 여전히 ~15° 정밀도로 복원 — 자기광학 커플링이 방위각을 $V$ 로 전달.
- 정보-풍부 범위: 6선일 때 $\log\tau \in [-2.6, 0.2]$, 2선일 때 $\log\tau < -0.4$ 로 축소.